# General Normal Forward Example

Define a symbolic general problem inside the notebook with `ProblemBuilder`, fix the inverse-style coefficients to known values, generate labels, then train KKT-HardNet in forward mode.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np

from kkthn.problem_builder import ProblemBuilder
from kkthn.problems import apply_label_noise, generate_labels_with_slsqp, write_json
from kkthn.projection import ProjectionSettings
from kkthn.training import KKTTrainConfig, train_kkt_hardnet


def make_run_dir(name):
    root = Path.cwd().resolve()
    notebook_dir = root if root.name == "notebooks" else root / "notebooks"
    run_dir = notebook_dir / "_runs" / name / time.strftime("%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def make_train_config(config, proj):
    return KKTTrainConfig(
        epochs=int(config["epochs"]),
        batch_size=int(config["batch_size"]),
        learning_rate=float(config["learning_rate"]),
        train_frac=float(config["train_frac"]),
        hidden_size=int(config["hidden_size"]),
        hidden_layers=int(config["hidden_layers"]),
        seed=int(config["seed"]),
        dtype=str(config["dtype"]),
        print_every=int(config["print_every"]),
        projection=ProjectionSettings(**proj),
    )


In [ ]:
DATA = {
    "type": "general_normal",
    "mode": "forward",
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "x_L": [-1.0, -1.0],
    "x_U": [1.0, 1.0],
    "inv_param": ["a0", "a1"],
    "inv_param_label": [1.0, -1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


In [ ]:
def build_normal_problem(data):
    builder = ProblemBuilder(y_bound=4.0)
    x = builder.add_parameter(["x1", "x2"])
    theta = builder.add_inverse_parameter(data["inv_param"])
    y = builder.add_variable(["y1", "y2", "y3"])

    builder.objective = 0.5 * (y.y1**2 + y.y2**2 + y.y3**2)
    builder.constraints.add(
        theta.a0 * y.y1 + y.y2 - x.x1 == 0,
        y.y2 - theta.a1 * y.y3 - x.x2 == 0,
        y.y1**2 + y.y3**2 <= 2.0,
    )
    builder.bounds.set(lower=-4.0, upper=4.0)
    return builder


In [ ]:
run_dir = make_run_dir("general_normal_forward")
builder = build_normal_problem(DATA)
inverse_values = np.asarray(DATA["inv_param_label"], dtype=np.float64)
model, problem_metadata = builder.build_model(train_inverse=False, inverse_values=inverse_values)

rng = np.random.default_rng(DATA["seed"])
X = rng.uniform(DATA["x_L"], DATA["x_U"], size=(DATA["num_samples"], len(builder.parameter_names)))
Y, label_metadata = generate_labels_with_slsqp(model, X)
Y, label_metadata = apply_label_noise(Y, DATA, label_metadata)
metadata = {
    "mode": "forward",
    "problem": problem_metadata,
    "inverse_parameter_names": builder.inverse_parameter_names,
    "inverse_parameter_labels": inverse_values.tolist(),
    **label_metadata,
}

write_json(run_dir / "problem.json", problem_metadata)
write_json(run_dir / "data.json", DATA)
write_json(run_dir / "config.json", CONFIG)
write_json(run_dir / "proj.json", PROJ)
write_json(run_dir / "label_metadata.json", metadata)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(json.dumps(problem_metadata, indent=2)[:2000])

result = train_kkt_hardnet(
    model=model,
    X=X,
    Y=Y,
    cfg=make_train_config(CONFIG, PROJ),
    output_dir=run_dir,
    metadata=metadata,
)


In [ ]:
summary = {
    "output_dir": result["output_dir"],
    "dims": result["dims"],
    "final": result["final"],
    "component_time_percent": result["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))
